# 01 · Machine Learning distribuido con Spark MLlib

Entrenamos un modelo de **clasificación** sobre datos generados de forma
distribuida, usando **Spark ML** (la librería de ML que corre sobre el clúster).

No hace falta descargar nada: los datos se crean dentro del propio clúster.


## 1. Sesión de Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("01-ml-spark")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-driver")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("Spark", spark.version, "| App UI:", spark.sparkContext.uiWebUrl)

## 2. Generar un dataset sintético

Creamos ~200.000 filas con 4 variables numéricas y una etiqueta binaria que
depende de ellas (con algo de ruido). Todo se genera en paralelo en el clúster.


In [ ]:
from pyspark.sql import functions as F

N = 200_000
datos = (
    spark.range(N)
    .withColumn("x1", F.randn(seed=1))
    .withColumn("x2", F.randn(seed=2))
    .withColumn("x3", F.randn(seed=3))
    .withColumn("x4", F.randn(seed=4))
)

# Regla latente + ruido -> etiqueta 0/1
score = (
    1.5 * F.col("x1") - 2.0 * F.col("x2")
    + 0.5 * F.col("x3") + F.randn(seed=5) * 0.5
)
datos = datos.withColumn("label", (score > 0).cast("int")).drop("id")

datos.show(5)
datos.groupBy("label").count().show()

## 3. Preparar features y partición train/test

`VectorAssembler` empaqueta las columnas en un único vector de características,
el formato que espera Spark ML.


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["x1", "x2", "x3", "x4"],
    outputCol="features",
)
dataset = assembler.transform(datos).select("features", "label")

train, test = dataset.randomSplit([0.8, 0.2], seed=42)
train.cache()
print("Train:", train.count(), "| Test:", test.count())

## 4. Entrenar un Random Forest

El entrenamiento se ejecuta de forma **distribuida** en los workers. Observa la
actividad en la **Spark App UI (puerto 4040)**.


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=6,
    seed=42,
)
modelo = rf.fit(train)
print("Modelo entrenado. Arboles:", modelo.getNumTrees)

## 5. Evaluar el modelo (AUC y exactitud)

In [ ]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

pred = modelo.transform(test)

auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(pred)

acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
).evaluate(pred)

print(f"AUC      : {auc:.4f}")
print(f"Exactitud: {acc:.4f}")
pred.select("label", "prediction", "probability").show(5, truncate=False)

## 6. Importancia de las variables

Confirmamos que el modelo ha "descubierto" que `x1` y `x2` son las que más pesan
(coherente con la regla que usamos para generar los datos).


In [ ]:
import pandas as pd

importancias = pd.DataFrame({
    "variable": ["x1", "x2", "x3", "x4"],
    "importancia": modelo.featureImportances.toArray(),
}).sort_values("importancia", ascending=False)
importancias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.barh(importancias["variable"], importancias["importancia"])
plt.gca().invert_yaxis()
plt.title("Importancia de variables (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
# spark.stop()  # descomenta al terminar